In [2]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [3]:
from tools.mongodbtools import(
    fetchUserProfile,
    fetchUserWallet,
    fetchUserBets,
    fetchUserPortfolio,
    fetchMatch
)
from rag.retriever import retrieveChunks
import logging

In [4]:
async def contextAdder(state):
    updates={}
    userId=state.get("user-id","")
    ctx=state.get("context",{})

    if userId and userId!="system":
        try:
            profile=await fetchUserProfile(userId)
            updates["user-profile"]=profile or {}
        except Exception as e:
            logger.warning(f"user data fetch failed {e}")
            updates["user-profile"]={}
        try:
            wallet=await fetchUserWallet(userId)
            updates["user-wallet"]=wallet or {}
        except Exception as e:
            logger.warning(f"user wallet fetch failed {e}")
            updates["user-wallet"]={}
        try:
            bets=await fetchUserBets(userId,limit=10)
            updates["user-recent-bets"]=bets or []
        except Exception as e:
            logger.warning(f"user bets fetch failed {e}")
            updates["user-recent-bets"]=[]
        try:
            portfolio=await fetchUserPortfolio(userId)
            updates["user-portfolio"]=portfolio or {}
        except Exception as e:
            logger.warning(f"user portfolio fetch failed {e}")
            updates["user-portfolio"]={}
    
    matchId=ctx.get("matchId","")
    if matchId:
        try:
            match=await fetchMatch(matchId)
            if match:
                updates["match-data"]=match
                updates["live-score"]=match.get("liveScore",{}) or {}
        except Exception as e:
            logger.warning(f"match data fetch failed {e}")
    
    query=state.get("query","")
    if query:
        try:
            chunks=await retrieveChunks(query,topK=5)
            updates["rag-chunks"]=chunks
        except Exception as e:
            updates["rag-chunks"]=[]
    return updates